# Transcient Light curves in DP1 ECDFS

In [ ]:
%matplotlib inline

# Using fastdb_client

This example is for running in Perlmutter.  You may be able to easily convert it for somewhere else.

Note that this example notebook was built when `fastdb-rknop-dev.lbl.gov` had the ELAsTiCC2 data loaded into it.  It's possible at some later time that you will not get the same results.  When a more stable version of fastdb exists, I will update this example notebook.

## What is fastdb_client

It's a python program useful for connecting to instances of FASTDB.  At some level, it's just a wrapper around python `requests`, but it handles some other stuff you wouldn't want to handle manually.  For example, logging into FASTDB is not just a matter of sending a password over the net, but is a complicated handshaking of public key cryptography (created by somebody trying to be paranoid to make sure the password itself never needed to be sent over the network from your machine to the server machine).  The fastdb_client takes care of all of that for you.  It has the ability to do some format parsing of responses from the web server.  Finally, it has a built-in "sleep and retry" cycle, where if the connection to the FASTDB server fails for some reason (which does happen--- this is the Internet, after all), it will sleep a few seconds and retry, repeating that (by default) five times before giving up for good.  This can greatly increase the reliability of scripts that use the fastdb_client, saving you from having to implement your own error catching and retry loop.

## Getting an account

To use FASTDB, you need an account on the instance of the FASTDB web server you want to interact with.  As of this writing, a couple exist.  The only really public one is `https://fastdb-dp1.lbl.gov`, though as of this writing the code running there is a bit out of date so won't entirely work with this notebook.  The only other one that exists is `https://fastdb-rknop-dev.lbl.gov`, and as its name suggests, it's a development server that may be down at any time, or that may have unexpected changes of contents and state at any time.  To get an account on ``fastdb-dp1.lbl.gov``, ask Rob Knop (via Discovery Alliance Slack) for an account; I will need the username you want for the account, the name you want displayed when you are logged in, and the email address associated with the account.  Once I tell you the account is created, go to the webserver in your browser, and click on "Request Password Reset".  Put in your username, and click "Email Password Resent Link".  Shortly thereafter, in your email you should find a message with a link to click on to reset your password.  (When your account is first created, it does not have a password; you can not log into it until you set one.)

Make sure to choose a <a href="http://rknop.net/password.html">good password</a>.  FASTDB is currently not using any TFA, and I'm really hoping we won't feel the need to go that way, as that will make a lot of life a lot more complicated.

### Setting up a convenience initialization file

You can always just use the url, username, and password every time you connect to fastdb.  However, it will probably make your life easier if you create a file `.fastdb.ini` (notice the period at the beginning) in your home directory.  This file will have contents like:

```
[rknop-dev]
url = https://fastdb-rknop-dev.lbl.gov
username = <your username>
password = <your password>

[dp1]
url = https://fastdb-dp1.lbl.gov
username = <your username>
password = <your password>

[production]
url = https://desc-fastdb.lbl.gov
username = <your username>
password = <your password>
```

Each block of lines starting with a line in brackets represents one FASTDB server you might want to connect to.  The name in brackets can be whatever you want; it will be wha tyou use in order to specify the server you want to conenct to.  Below each bracketed line, you need three lines, similar to the example above, which specify the URL of the web server, your username on that server, and your password on that server.  (As of this writing you do not want to include the `[production]` block because that server does not exist.)

**Important**: the fastdb client will refuse to use this `.ini` file if it's permissions aren't sufficiently paranoid.  After you've created the file, run
```
cd
chmod go -rwx .fastdb.ini
```
to make sure that only you can read it.

There are actually a number of other configuration options you can specify here, but for now they're not that important.  Talk to Rob, or read the `fastdb_client.py` source code if you're interested.

## Setting up your environment

`fastdb_client.py` requires libraries that are not yet in the standard desc time domain environment.  However, they are in the current dev environment.  On Perlmutter, you can get yourself in this environment with
```
source $CFS/lsst/groups/TD/setup_td_dev.sh
```

### Running from jupyter

Make sure to select the `desc-td-env-dev` kernel.  If that kernel isn't available, you can create it on Perlmutter with:
```
cd ~/.local/share/jupyter/kernels
mkdir desc-td-env-dev
```

In this directory, create two files.  The first one is named `td-env-dev.sh` and has conents:
```
#!/bin/bash

INST_DIR=/global/cfs/cdirs/lsst/groups/TD
source $INST_DIR/setup_td_dev.sh -k -c

if [ $# -gt 0 ] ; then
    exec python -m ipykernel $@
fi
```

the second one is `kernel.json` and has contents:
```
{
  "argv": [
    "/global/common/software/lsst/common/miniconda/kernels/td-env-dev.sh",
    "-f",
    "{connection_file}"
  ],
  "display_name": "desc-td-env-dev",
  "language": "python"
}
```


In [ ]:
# The FastDB client can be found on Perlmutter in the directory
#   /global/cfs/cdirs/lsst/groups/TD/SOFTWARE/fastdb_deployment/fastdb_client
# (The client there is actually a symbolic link to something else underneath
# fastdb_deployment.  If you know what you're doing, you may want to use a
# version of the client other than the one there, but by default that's the
# right one to use.)  Put this directory in your PYTHONPATH so it will be found.
import sys
sys.path.insert( 0, '/global/cfs/cdirs/lsst/groups/TD/SOFTWARE/fastdb_deployment/fastdb_client' )

#...and import!
from fastdb_client import FASTDBClient

# Also import other stuff we'll need
import io
import time
import pandas
import pandas as pd
import json
from matplotlib import pyplot
import matplotlib.pyplot as plt


In [ ]:
# Create your connection to fastdb.  This will log you in.
#
# There are two ways to do this.
# You can always just specify the url, username, and password, with 
#   something like:
#      fdb = FASTDBClient( 'https://fastdb-rknop-dev.lbl.gov', 'rknop', 'not_really_robs_password' )
#   where the three arguments are url, username, and password.  BE CAREFUL PUTTING PASSWORDS IN SCRIPTS,
#   and NEVER commit a script that has a password in it to a git archive anywhere!  If you do, immmediately
#   change your password.
# The other alternative is just to use the name you set up in your ~/.fastdb.ini file (described above).
#   that's what I do here.

#fdb = FASTDBClient( "rknop-dev" )
fdb = FASTDBClient( "dp1" )

##### HACK ALERT
fdb.retries = 0

## WEB API Endpoints

You use the fastdb client to connect to web API endpoints on the server.  All of these will eventually be documented, but for now, here are some examples of endpoints that you can hit.

In [ ]:
# /getprocvers lists all the known processing versions and processing version aliases. 
# (There are at most as many different processing versions as the list you get back,
# because one processing version may have multiple aliases.)
res = fdb.post( "/getprocvers" )
print( res )

In [ ]:
# You can get the lightcurve for an object if you know it's object ID with the /ltcv/getltcv 
# endpoint.  Give it the processing version and the diaobjectid as additional slash-separated
# url components.
t0 = time.perf_counter()
#ltcvdict = fdb.post( "/ltcv/getltcv/realtime/5181140" )
#ltcvdict = fdb.post( "/ltcv/getltcv/realtime/579578005207130164")
ltcvdict = fdb.post( "/ltcv/getltcv/dp1/609788805167188915")
print( f"/ltcv/getltcv took {time.perf_counter()-t0:.2f} seconds" )

# The dictionary you get back has keys 'status', 'objectinfo' and 'ltcv':
print( f"Got back ltcvdict with keys {ltcvdict.keys()}" )

objinfo = { k: v for k, v in ltcvdict.items() if k != 'ltcv' }
print( f"Object info: {json.dumps( objinfo, indent=4 )}" )

ltcv = pandas.DataFrame( ltcvdict['ltcv'] )
ltcv

In [ ]:
objinfo

In [ ]:
diaobjid = objinfo['objinfo']['diaobjectid']

In [ ]:
def PlotLightCurves(df,diaobjid = 0, numdet=0 ,figx= 16, figy=6):
    """
    """
    # Définir les couleurs et marqueurs pour chaque bande
    colors = {'u': 'blue','g':'green','r': 'red','i':'orange','z':'grey','y':'k'}
    markers = {'u': '^','g':'v','r': 'o','i':'s','z':'>','y':'<'}


    YMAX = df.describe().loc['max']['psfflux']
    YMIN = 0
    
    fig,ax = plt.subplots(1,1,figsize=(figx, figy))

    # Parcourir chaque bande unique
    for band in df['band'].unique():
        subset = df[df['band'] == band]
        ax.errorbar(
            subset['mjd'],
            subset['psfflux'],
            yerr=subset['psffluxerr'],
            fmt=markers[band],  # Marqueur
            color=colors[band],  # Couleur
            label=f'Band {band}',
            capsize=5  # Taille des barres d'erreur
        )
    # Ajouter des labels et une légende
    ax.set_ylim(YMIN,YMAX)
    ax.set_xlabel('MJD')
    ax.set_ylabel('PSF Flux')
    ax.set_title(f"psf flux, diaobjid = {diaobjid}, numdet = {numdet}")
    ax.legend()
    ax.grid(True)

    # Afficher le plot
    plt.show()


In [ ]:
PlotLightCurves(ltcv,diaobjid=diaobjid)

In [ ]:
# You can search for lightcurves with /objectsearch/<procver>
# This one requires you to pass some search data as a dict:

blah = fdb.retries
fdb.retries = 0
res = fdb.post( "/objectsearch/dp1", json={"ra":  53.1167, "dec": -27.8083, "radius": 600.,"noforced": True} )
fdb.retries = blah
objs = pd.DataFrame(res)
objs

# Note that this query can be kind of slow.  The slowest parts are where it gets the latest photometry and
# forced photometry.  You can make this query faster by adding "noforced": True to the dictionary.
# You won't get the latest forced photometry point back, but the query should return sooner.

In [ ]:
# Sélectionner les 10 lignes avec les valeurs les plus élevées de 'numdet'
top_numdet = objs.nlargest(20, 'numdet')

# Afficher le résultat
#print(top_10_numdet)

In [ ]:
for index, row in top_numdet.iterrows():
    diaobjid = row['diaobjectid']
    numdet = row['numdet']
    msg = f"/ltcv/getltcv/dp1/{diaobjid}"
    print(msg)

    t0 = time.perf_counter()
    ltcvdict = fdb.post(msg)
    print( f"/ltcv/getltcv took {time.perf_counter()-t0:.2f} seconds" )

    # The dictionary you get back has keys 'status', 'objectinfo' and 'ltcv':
    #print( f"Got back ltcvdict with keys {ltcvdict.keys()}" )

    objinfo = { k: v for k, v in ltcvdict.items() if k != 'ltcv' }
    #print( f"Object info: {json.dumps( objinfo, indent=4 )}" )
    
    #diaobjid = objinfo['objinfo']['diaobjectid']
    ltcv = pandas.DataFrame( ltcvdict['ltcv'] )


    #
    PlotLightCurves(ltcv,diaobjid=diaobjid,numdet=numdet)